In [ ]:
# import torch
# torch.cuda.is_available()



False

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# import os
# os.listdir("/content/drive/MyDrive")


['stubble_vision']

In [ ]:
# os.listdir("/content/drive/MyDrive/stubble_vision/data/raw")


['viirs_punjab.csv']

In [ ]:
import os
import pandas as pd

# Define raw data directory
CSV_DIR = "/content/drive/MyDrive/stubble_vision/data/raw"

# 1️⃣ Check directory exists
if not os.path.exists(CSV_DIR):
    raise FileNotFoundError(f"❌ Directory does not exist: {CSV_DIR}")

# 2️⃣ List files
files = os.listdir(CSV_DIR)
print("Files found:", files)

# 3️⃣ Filter CSV files
csv_files = [f for f in files if f.lower().endswith(".csv")]

# 4️⃣ Handle no-CSV case safely
if len(csv_files) == 0:
    raise FileNotFoundError(
        f"❌ No CSV files found in {CSV_DIR}. "
        "Please make sure your FIRMS CSV is inside data/raw."
    )

# 5️⃣ Load first CSV (or change index if needed)
csv_file = csv_files[0]
csv_path = os.path.join(CSV_DIR, csv_file)

df = pd.read_csv(csv_path)

print("✅ CSV loaded successfully!")
print("File name:", csv_file)
print("Shape (rows, columns):", df.shape)

# Optional sanity check
df.head()


Files found: ['viirs_punjab.csv']
✅ CSV loaded successfully!
File name: viirs_punjab.csv
Shape (rows, columns): (4353, 14)


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,28.35642,76.14301,335.23,0.52,0.41,2025-11-01,746,N,VIIRS,n,2.0NRT,304.11,2.04,D
1,28.35764,76.14528,331.44,0.52,0.41,2025-11-01,746,N,VIIRS,n,2.0NRT,302.41,3.06,D
2,29.09356,77.48702,333.13,0.44,0.38,2025-11-01,746,N,VIIRS,n,2.0NRT,302.25,3.27,D
3,29.20996,77.63349,338.85,0.43,0.38,2025-11-01,746,N,VIIRS,n,2.0NRT,301.33,1.84,D
4,29.24997,75.99911,332.38,0.51,0.41,2025-11-01,746,N,VIIRS,n,2.0NRT,301.17,3.41,D


In [ ]:
df.head()

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,28.35642,76.14301,335.23,0.52,0.41,2025-11-01,746,N,VIIRS,n,2.0NRT,304.11,2.04,D
1,28.35764,76.14528,331.44,0.52,0.41,2025-11-01,746,N,VIIRS,n,2.0NRT,302.41,3.06,D
2,29.09356,77.48702,333.13,0.44,0.38,2025-11-01,746,N,VIIRS,n,2.0NRT,302.25,3.27,D
3,29.20996,77.63349,338.85,0.43,0.38,2025-11-01,746,N,VIIRS,n,2.0NRT,301.33,1.84,D
4,29.24997,75.99911,332.38,0.51,0.41,2025-11-01,746,N,VIIRS,n,2.0NRT,301.17,3.41,D


In [ ]:
df.columns

Index(['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date',
       'acq_time', 'satellite', 'instrument', 'confidence', 'version',
       'bright_ti5', 'frp', 'daynight'],
      dtype='object')

In [ ]:
# map VIIRS confidence to numeric
confidence_map = {
    "l": 30,
    "n": 70,
    "h": 100
}

df["confidence_num"] = df["confidence"].map(confidence_map)

# create fire / no-fire label
df["label"] = (df["confidence_num"] >= 70).astype(int)

# verify
df[["confidence", "confidence_num", "label"]].head()


,confidence,confidence_num,label
0,n,70,1
1,n,70,1
2,n,70,1
3,n,70,1
4,n,70,1


In [ ]:
df["label"].value_counts()


,count
label,
1,4122
0,231


In [ ]:
# columns we want to keep
keep_cols = [
    "latitude",
    "longitude",
    "bright_ti4",
    "bright_ti5",
    "frp",
    "acq_date",
    "daynight",
    "confidence_num",
    "label"
]

df_clean = df[keep_cols].copy()

df_clean.head()


,latitude,longitude,bright_ti4,bright_ti5,frp,acq_date,daynight,confidence_num,label
0,28.35642,76.14301,335.23,304.11,2.04,2025-11-01,D,70,1
1,28.35764,76.14528,331.44,302.41,3.06,2025-11-01,D,70,1
2,29.09356,77.48702,333.13,302.25,3.27,2025-11-01,D,70,1
3,29.20996,77.63349,338.85,301.33,1.84,2025-11-01,D,70,1
4,29.24997,75.99911,332.38,301.17,3.41,2025-11-01,D,70,1


In [ ]:
df_clean.isnull().sum()


,0
latitude,0
longitude,0
bright_ti4,0
bright_ti5,0
frp,0
acq_date,0
daynight,0
confidence_num,0
label,0


In [ ]:
df_clean["label"].value_counts()


,count
label,
1,4122
0,231


In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.30,
    random_state=42,
    stratify=df_clean["label"]
)

print("Train shape:", train_df.shape)
print("Temp shape:", temp_df.shape)


Train shape: (3047, 9)
Temp shape: (1306, 9)


In [ ]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)


Validation shape: (653, 9)
Test shape: (653, 9)


In [ ]:
print("Train labels:\n", train_df["label"].value_counts())
print("\nValidation labels:\n", val_df["label"].value_counts())
print("\nTest labels:\n", test_df["label"].value_counts())


Train labels:
 label
1    2885
0     162
Name: count, dtype: int64

Validation labels:
 label
1    619
0     34
Name: count, dtype: int64

Test labels:
 label
1    618
0     35
Name: count, dtype: int64


In [ ]:
BASE_DIR = "/content/drive/MyDrive/stubble_vision/data/processed"

train_df.to_csv(f"{BASE_DIR}/train.csv", index=False)
val_df.to_csv(f"{BASE_DIR}/val.csv", index=False)
test_df.to_csv(f"{BASE_DIR}/test.csv", index=False)

print("✅ Train / Validation / Test CSV files saved")


✅ Train / Validation / Test CSV files saved


In [ ]:
import os
os.listdir(BASE_DIR)


['train.csv', 'val.csv', 'test.csv']

In [ ]:
# features we will use
feature_cols = [
    "bright_ti4",
    "bright_ti5",
    "frp"
]

X_train = train_df[feature_cols]
y_train = train_df["label"]

X_val = val_df[feature_cols]
y_val = val_df["label"]

X_test = test_df[feature_cols]
y_test = test_df["label"]

print("Features selected:", feature_cols)


Features selected: ['bright_ti4', 'bright_ti5', 'frp']


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=10000)

model.fit(X_train, y_train)

print("✅ Model training complete")


✅ Model training complete


In [ ]:
val_accuracy = model.score(X_val, y_val)
print("Validation Accuracy:", val_accuracy)


Validation Accuracy: 0.9433384379785605


In [ ]:
test_accuracy = model.score(X_test, y_test)
print("Test Accuracy:", test_accuracy)


Test Accuracy: 0.9418070444104135


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)


In [ ]:
y_test_pred = model.predict(X_test)


In [ ]:
print(classification_report(y_test, y_test_pred, target_names=["No Fire", "Fire"]))


              precision    recall  f1-score   support

     No Fire       0.38      0.14      0.21        35
        Fire       0.95      0.99      0.97       618

    accuracy                           0.94       653
   macro avg       0.67      0.56      0.59       653
weighted avg       0.92      0.94      0.93       653



In [ ]:
!pip install -q earthengine-api


In [ ]:
import ee
ee.Authenticate()

True

In [ ]:
import ee

ee.Initialize(project="stubble-vision-ee")
print("✅ Earth Engine initialized successfully")


✅ Earth Engine initialized successfully


In [ ]:
point = ee.Geometry.Point([76.143, 28.356])

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR")
    .filterBounds(point)
    .filterDate("2023-10-01", "2023-10-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

print("Number of Sentinel-2 images:", s2.size().getInfo())


/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for COPERNICUS/S2_SR! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR

  warnings.warn(warning, category=DeprecationWarning)


Number of Sentinel-2 images: 5


In [ ]:
import ee
from datetime import datetime, timedelta

# Fire info (example)
lat, lon = 28.356, 76.143
fire_date = "2025-11-01"

point = ee.Geometry.Point([lon, lat])
fire_dt = datetime.strptime(fire_date, "%Y-%m-%d")


In [ ]:
# Time windows
pre_start  = (fire_dt - timedelta(days=30)).strftime("%Y-%m-%d")
pre_end    = (fire_dt - timedelta(days=5)).strftime("%Y-%m-%d")

post_start = (fire_dt + timedelta(days=1)).strftime("%Y-%m-%d")
post_end   = (fire_dt + timedelta(days=10)).strftime("%Y-%m-%d")


In [ ]:
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(point)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)


In [ ]:
pre_fire = (
    s2.filterDate(pre_start, pre_end)
      .sort("CLOUDY_PIXEL_PERCENTAGE")
      .first()
)

print("Pre-fire image found:", pre_fire is not None)


Pre-fire image found: True


In [ ]:
post_fire = (
    s2.filterDate(post_start, post_end)
      .sort("CLOUDY_PIXEL_PERCENTAGE")
      .first()
)

print("Post-fire image found:", post_fire is not None)


Post-fire image found: True


In [ ]:
# Select NIR and SWIR bands
pre_bands  = pre_fire.select(["B8", "B12"])
post_bands = post_fire.select(["B8", "B12"])


In [ ]:
# Compute NBR for pre-fire image
nbr_pre = pre_bands.normalizedDifference(["B8", "B12"]).rename("NBR_pre")

# Compute NBR for post-fire image
nbr_post = post_bands.normalizedDifference(["B8", "B12"]).rename("NBR_post")


In [ ]:
# dNBR = pre - post
dnbr = nbr_pre.subtract(nbr_post).rename("dNBR")


In [ ]:
# Fire location buffer (meters)
buffer_m = 500   # you can later test 500m, 1km, etc.

fire_area = ee.Geometry.Point([lon, lat]).buffer(buffer_m)


In [ ]:
dnbr_mean = dnbr.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=fire_area,
    scale=20,        # Sentinel-2 SWIR resolution
    maxPixels=1e9
)

dnbr_value = dnbr_mean.get("dNBR").getInfo()
print("Mean dNBR:", dnbr_value)



Mean dNBR: -0.02982601876621879


In [ ]:
def classify_severity(dnbr):
    if dnbr < 0.10:
        return "Unburned"
    elif dnbr < 0.27:
        return "Low"
    elif dnbr < 0.44:
        return "Moderate"
    else:
        return "High"

severity = classify_severity(dnbr_value)
print("Fire severity:", severity)


Fire severity: Unburned


In [ ]:
fire_events = df[df["label"] == 1].copy()
print("Total fire events:", len(fire_events))


Total fire events: 4122


In [ ]:
from datetime import datetime, timedelta
import ee

def compute_fire_severity(lat, lon, fire_date, buffer_m=500):
    try:
        point = ee.Geometry.Point([lon, lat])
        fire_dt = datetime.strptime(fire_date, "%Y-%m-%d")

        # ---- Time windows ----
        pre_start  = (fire_dt - timedelta(days=30)).strftime("%Y-%m-%d")
        pre_end    = (fire_dt - timedelta(days=5)).strftime("%Y-%m-%d")
        post_start = (fire_dt + timedelta(days=1)).strftime("%Y-%m-%d")
        post_end   = (fire_dt + timedelta(days=10)).strftime("%Y-%m-%d")

        # ---- Sentinel-2 collection ----
        s2 = (
            ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(point)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        )

        pre = s2.filterDate(pre_start, pre_end).sort("CLOUDY_PIXEL_PERCENTAGE").first()
        post = s2.filterDate(post_start, post_end).sort("CLOUDY_PIXEL_PERCENTAGE").first()

        if pre is None or post is None:
            return None, "No Image"

        # ---- Compute NBR ----
        nbr_pre = pre.normalizedDifference(["B8", "B12"])
        nbr_post = post.normalizedDifference(["B8", "B12"])

        # ---- Compute dNBR (IMPORTANT: rename band) ----
        dnbr = nbr_pre.subtract(nbr_post).rename("dNBR")

        # ---- Buffer around fire ----
        area = point.buffer(buffer_m)

        # ---- Reduce to mean dNBR ----
        dnbr_dict = dnbr.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=area,
            scale=20,
            maxPixels=1e9,
            bestEffort=True
        ).getInfo()

        if dnbr_dict is None or "dNBR" not in dnbr_dict:
            return None, "No Data"

        dval = dnbr_dict["dNBR"]

        # ---- Severity classification ----
        if dval < 0.10:
            severity = "Unburned / Recovered"
        elif dval < 0.27:
            severity = "Low"
        elif dval < 0.44:
            severity = "Moderate"
        else:
            severity = "High"

        return dval, severity

    except Exception as e:
        return None, f"Error: {e}"


In [ ]:
results = []

fire_events = df[df["label"] == 1].copy()

for idx, row in fire_events.head(100).iterrows():
    dnbr_val, severity = compute_fire_severity(
        row["latitude"],
        row["longitude"],
        row["acq_date"]
    )

    results.append({
        "index": idx,
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "fire_date": row["acq_date"],
        "dNBR": dnbr_val,
        "severity": severity
    })

    print(f"[{idx}] dNBR={dnbr_val}, severity={severity}")


[0] dNBR=-0.030837065833754448, severity=Unburned / Recovered
[1] dNBR=-0.015824437075709116, severity=Unburned / Recovered
[2] dNBR=-0.020709029757102825, severity=Unburned / Recovered
[3] dNBR=0.022080862919853943, severity=Unburned / Recovered
[4] dNBR=0.2531020275872906, severity=Low
[5] dNBR=0.1688115554359553, severity=Low
[6] dNBR=0.24453826009435156, severity=Low
[7] dNBR=0.1939132397822736, severity=Low
[8] dNBR=0.19723350028840775, severity=Low
[9] dNBR=0.20946787497002786, severity=Low
[10] dNBR=0.21402027758903797, severity=Low
[11] dNBR=0.20884207927564233, severity=Low
[12] dNBR=0.16129380560328016, severity=Low
[13] dNBR=0.10734013031150426, severity=Low
[15] dNBR=0.1435667554676517, severity=Low
[16] dNBR=0.21710554129661225, severity=Low
[17] dNBR=0.24813933773572855, severity=Low
[18] dNBR=0.2980611675632353, severity=Moderate
[19] dNBR=0.11648665530860162, severity=Low
[20] dNBR=0.17683907565882556, severity=Low
[21] dNBR=0.37225983944268565, severity=Moderate
[22] d

In [ ]:
import ee
import pandas as pd
from datetime import datetime, timedelta

ee.Initialize(project="stubble-vision-ee")


In [ ]:
fire_events = df[df["label"] == 1].copy().reset_index(drop=True)
print("Total fire events:", len(fire_events))


Total fire events: 4122


In [ ]:
def compute_fire_severity(lat, lon, fire_date, buffer_m=500):
    try:
        point = ee.Geometry.Point([lon, lat])
        fire_dt = datetime.strptime(fire_date, "%Y-%m-%d")

        pre_start  = (fire_dt - timedelta(days=30)).strftime("%Y-%m-%d")
        pre_end    = (fire_dt - timedelta(days=5)).strftime("%Y-%m-%d")
        post_start = (fire_dt + timedelta(days=1)).strftime("%Y-%m-%d")
        post_end   = (fire_dt + timedelta(days=10)).strftime("%Y-%m-%d")

        s2 = (
            ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(point)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        )

        pre = s2.filterDate(pre_start, pre_end).sort("CLOUDY_PIXEL_PERCENTAGE").first()
        post = s2.filterDate(post_start, post_end).sort("CLOUDY_PIXEL_PERCENTAGE").first()

        if pre is None or post is None:
            return None, "No Image"

        nbr_pre = pre.normalizedDifference(["B8", "B12"])
        nbr_post = post.normalizedDifference(["B8", "B12"])

        dnbr = nbr_pre.subtract(nbr_post).rename("dNBR")

        area = point.buffer(buffer_m)

        dnbr_dict = dnbr.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=area,
            scale=20,
            maxPixels=1e9,
            bestEffort=True
        ).getInfo()

        if dnbr_dict is None or "dNBR" not in dnbr_dict:
            return None, "No Data"

        dval = dnbr_dict["dNBR"]

        if dval < 0.10:
            sev = "Unburned / Recovered"
        elif dval < 0.27:
            sev = "Low"
        elif dval < 0.44:
            sev = "Moderate"
        else:
            sev = "High"

        return dval, sev

    except Exception as e:
        return None, f"Error"


In [ ]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/stubble_vision/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Results directory ready:", OUTPUT_DIR)


✅ Results directory ready: /content/drive/MyDrive/stubble_vision/results


In [ ]:
BATCH_SIZE = 50          # do NOT increase
START_INDEX = 0          # change if resuming
END_INDEX = len(fire_events)

OUTPUT_DIR = "/content/drive/MyDrive/stubble_vision/results"


In [ ]:
results = []

for i in range(START_INDEX, END_INDEX):
    row = fire_events.iloc[i]

    dnbr_val, severity = compute_fire_severity(
        row["latitude"],
        row["longitude"],
        row["acq_date"]
    )

    results.append({
        "fire_id": i,
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "fire_date": row["acq_date"],
        "dNBR": dnbr_val,
        "severity": severity
    })

    # Progress log
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1} / {END_INDEX}")

    # Save batch
    if (i + 1) % BATCH_SIZE == 0:
        batch_df = pd.DataFrame(results)
        batch_path = f"{OUTPUT_DIR}/severity_batch_{i+1}.csv"
        batch_df.to_csv(batch_path, index=False)
        print(f"Saved {batch_path}")
        results = []  # clear memory


Processed 10 / 4122
Processed 20 / 4122
Processed 30 / 4122
Processed 40 / 4122
Processed 50 / 4122
Saved /content/drive/MyDrive/stubble_vision/results/severity_batch_50.csv
Processed 60 / 4122
Processed 70 / 4122
Processed 80 / 4122
Processed 90 / 4122
Processed 100 / 4122
Saved /content/drive/MyDrive/stubble_vision/results/severity_batch_100.csv
Processed 110 / 4122
Processed 120 / 4122
Processed 130 / 4122
Processed 140 / 4122
Processed 150 / 4122
Saved /content/drive/MyDrive/stubble_vision/results/severity_batch_150.csv
Processed 160 / 4122
Processed 170 / 4122
Processed 180 / 4122
Processed 190 / 4122
Processed 200 / 4122
Saved /content/drive/MyDrive/stubble_vision/results/severity_batch_200.csv
Processed 210 / 4122
Processed 220 / 4122
Processed 230 / 4122
Processed 240 / 4122
Processed 250 / 4122
Saved /content/drive/MyDrive/stubble_vision/results/severity_batch_250.csv
Processed 260 / 4122
Processed 270 / 4122
Processed 280 / 4122
Processed 290 / 4122
Processed 300 / 4122
Saved

Processed 2890 / 4122
Processed 2900 / 4122
Saved /content/drive/MyDrive/stubble_vision/results/severity_batch_2900.csv
Processed 2910 / 4122
Processed 2920 / 4122
Processed 2930 / 4122
Processed 2940 / 4122
Processed 2950 / 4122
Saved /content/drive/MyDrive/stubble_vision/results/severity_batch_2950.csv
Processed 2960 / 4122
Processed 2970 / 4122
Processed 2980 / 4122
Processed 2990 / 4122
Processed 3000 / 4122
Saved /content/drive/MyDrive/stubble_vision/results/severity_batch_3000.csv
Processed 3010 / 4122
Processed 3020 / 4122
Processed 3030 / 4122
Processed 3040 / 4122
Processed 3050 / 4122
Saved /content/drive/MyDrive/stubble_vision/results/severity_batch_3050.csv
Processed 3060 / 4122
Processed 3070 / 4122
Processed 3080 / 4122
Processed 3090 / 4122
Processed 3100 / 4122
Saved /content/drive/MyDrive/stubble_vision/results/severity_batch_3100.csv
Processed 3110 / 4122
Processed 3120 / 4122
Processed 3130 / 4122
Processed 3140 / 4122
Processed 3150 / 4122
Saved /content/drive/MyDri

In [ ]:
if results:
    final_df = pd.DataFrame(results)
    final_path = f"{OUTPUT_DIR}/severity_final.csv"
    final_df.to_csv(final_path, index=False)
    print(f"Saved {final_path}")


Saved /content/drive/MyDrive/stubble_vision/results/severity_final.csv


In [ ]:
import os

RESULTS_DIR = "/content/drive/MyDrive/stubble_vision/results"
files = sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith(".csv")])

print("Number of result files:", len(files))
files[:]


Number of result files: 83


['severity_batch_100.csv',
 'severity_batch_1000.csv',
 'severity_batch_1050.csv',
 'severity_batch_1100.csv',
 'severity_batch_1150.csv',
 'severity_batch_1200.csv',
 'severity_batch_1250.csv',
 'severity_batch_1300.csv',
 'severity_batch_1350.csv',
 'severity_batch_1400.csv',
 'severity_batch_1450.csv',
 'severity_batch_150.csv',
 'severity_batch_1500.csv',
 'severity_batch_1550.csv',
 'severity_batch_1600.csv',
 'severity_batch_1650.csv',
 'severity_batch_1700.csv',
 'severity_batch_1750.csv',
 'severity_batch_1800.csv',
 'severity_batch_1850.csv',
 'severity_batch_1900.csv',
 'severity_batch_1950.csv',
 'severity_batch_200.csv',
 'severity_batch_2000.csv',
 'severity_batch_2050.csv',
 'severity_batch_2100.csv',
 'severity_batch_2150.csv',
 'severity_batch_2200.csv',
 'severity_batch_2250.csv',
 'severity_batch_2300.csv',
 'severity_batch_2350.csv',
 'severity_batch_2400.csv',
 'severity_batch_2450.csv',
 'severity_batch_250.csv',
 'severity_batch_2500.csv',
 'severity_batch_2550.cs

In [ ]:
import pandas as pd

dfs = []

for f in files:
    path = os.path.join(RESULTS_DIR, f)
    dfs.append(pd.read_csv(path))

severity_all = pd.concat(dfs, ignore_index=True)

print("Merged severity shape:", severity_all.shape)
severity_all.head()


Merged severity shape: (4122, 6)


,fire_id,latitude,longitude,fire_date,dNBR,severity
0,50,29.90912,77.87334,2025-11-01,0.041275,Unburned / Recovered
1,51,29.91256,77.87257,2025-11-01,0.052702,Unburned / Recovered
2,52,29.98193,76.76251,2025-11-01,0.065801,Unburned / Recovered
3,53,29.98950,75.82626,2025-11-01,0.385759,Moderate
4,54,30.00345,76.08458,2025-11-01,0.428874,Moderate


In [ ]:
severity_all["severity"].value_counts()


,count
severity,
Moderate,1466
Low,1410
High,647
Unburned / Recovered,593
Error,6


In [ ]:
# make sure severity_all exists
print("severity_all shape:", severity_all.shape)

# filter only valid severity classes
valid_severity = severity_all[
    severity_all["severity"].isin(
        ["Unburned / Recovered", "Low", "Moderate", "High"]
    )
].copy()

print("Valid severity rows:", len(valid_severity))


severity_all shape: (4122, 6)
Valid severity rows: 4116


In [ ]:
severity_counts = valid_severity["severity"].value_counts()
severity_percent = severity_counts / severity_counts.sum() * 100

severity_summary = pd.DataFrame({
    "count": severity_counts,
    "percentage (%)": severity_percent.round(2)
})

severity_summary


,count,percentage (%)
severity,,
Moderate,1466,35.62
Low,1410,34.26
High,647,15.72
Unburned / Recovered,593,14.41


In [ ]:
# pick first Low severity fire
sample_fire = valid_severity[valid_severity["severity"] == "Low"].iloc[0]

lat = sample_fire["latitude"]
lon = sample_fire["longitude"]
fire_date = sample_fire["fire_date"]

lat, lon, fire_date


(np.float64(29.7423), np.float64(74.42926), '2025-11-01')

In [ ]:
from datetime import datetime, timedelta
import ee

point = ee.Geometry.Point([lon, lat])
fire_dt = datetime.strptime(fire_date, "%Y-%m-%d")

pre_start  = (fire_dt - timedelta(days=30)).strftime("%Y-%m-%d")
pre_end    = (fire_dt - timedelta(days=5)).strftime("%Y-%m-%d")
post_start = (fire_dt + timedelta(days=1)).strftime("%Y-%m-%d")
post_end   = (fire_dt + timedelta(days=10)).strftime("%Y-%m-%d")

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(point)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

pre = s2.filterDate(pre_start, pre_end).sort("CLOUDY_PIXEL_PERCENTAGE").first()
post = s2.filterDate(post_start, post_end).sort("CLOUDY_PIXEL_PERCENTAGE").first()


In [ ]:
nbr_pre = pre.normalizedDifference(["B8", "B12"])
nbr_post = post.normalizedDifference(["B8", "B12"])
dnbr = nbr_pre.subtract(nbr_post).rename("dNBR")


In [ ]:
# Visualization styles
nbr_vis = {
    "min": -0.5,
    "max": 0.8,
    "palette": ["brown", "yellow", "green"]
}

dnbr_vis = {
    "min": -0.1,
    "max": 0.6,
    "palette": ["green", "yellow", "orange", "red"]
}


In [ ]:
import geemap

Map = geemap.Map(center=[lat, lon], zoom=12)

Map.addLayer(nbr_pre, nbr_vis, "NBR Pre-Fire")
Map.addLayer(nbr_post, nbr_vis, "NBR Post-Fire")
Map.addLayer(dnbr, dnbr_vis, "dNBR (Severity)")

Map.addLayer(point, {"color": "blue"}, "Fire Location")

Map


Map(center=[np.float64(29.7423), np.float64(74.42926)], controls=(WidgetControl(options=['position', 'transpar…